# Medallion demo, part 3: clean up

Undo what `02-medallion-pipeline.ipynb` did, in steps, in the Azure DuckLake of `azure.config.json`. Every step has its own switch in the first cell and **nothing happens until `DRY_RUN = False`**: with the default the notebook only reports what it would do.

| Step | Switch | What it does |
|---|---|---|
| 1 | `RESET_SCHEMA_EVOLUTION` | Removes the column `loyalty_tier` again from `bronze.customers`, `silver.customer`, `silver.customer_change` and `gold.dim_customer` (`ALTER TABLE ... DROP COLUMN`), so the layers look as they did before 15 September. |
| 2 | `RESET_LAYERS` | Drops the schemas `bronze`, `silver`, `gold` and `ops` with everything in them. The demo can be loaded from scratch afterwards. |
| 3 | `DELETE_BRONZE_FILES` | Deletes the Parquet files in the folder `bronze` of the storage account (`bronzePrefix` in `azure.config.json`). Afterwards `01-medallion-generator.ipynb` has to be run again. |
| 4 | `EXPIRE_SNAPSHOTS` | Expires **all** old DuckLake snapshots and deletes the data files that no snapshot needs any more. This frees the storage of dropped or replaced tables, but it also removes the time travel history of **every** table in the catalog. |

**Note on step 1.** DuckLake never loses data by itself: a dropped column is only removed from the current schema; older snapshots still show it (time travel) until step 4 expires them. Data that was only held in the dropped column
is gone from the current tables. To fill it again, load the days from 15 September on once more (`02-medallion-pipeline.ipynb`, backfill): Bronze re-adds the column and the days' rows, Silver and Gold recompute the touched customers.

**Azure.** This notebook belongs to the Azure DuckLake (the one that aantje creates with *Create Azure DuckLake...*). It reads the connection data from
`azure.config.json` next to it (storage account, container, PostgreSQL server, sign-in mode; no secrets) through `azure_lake.py`. Fill the file in once; all three Azure notebooks use it.
The PostgreSQL password comes from the environment variable `OOG_AZURE_PG_PASSWORD` or a prompt; the storage account is reached with your Microsoft Entra sign-in (a code that you enter in the browser, or a service principal). The public IP address of this machine has to be allowed in the firewall of the PostgreSQL server (aantje: *Allow My IP*).

In [ ]:
# Once per kernel: the Microsoft Entra sign-in and the client that deletes the Bronze files. If the import fails afterwards, restart the kernel and run the cells again.
%pip install -q azure-identity azure-storage-file-datalake

In [ ]:
import azure_lake

DRY_RUN = True                    # True: only report. False: really do what the switches below say.

RESET_SCHEMA_EVOLUTION = True     # step 1: drop the evolved column loyalty_tier again
RESET_LAYERS = False              # step 2: drop the schemas bronze, silver, gold, ops
DELETE_BRONZE_FILES = False       # step 3: delete the Bronze Parquet files in Azure Storage
EXPIRE_SNAPSHOTS = False          # step 4: expire all snapshots and delete unreferenced files (affects ALL tables of the catalog)

EVOLVED_COLUMNS = {                      # table -> columns that appeared by schema evolution
    ("bronze", "customers"): ["loyalty_tier"],
    ("silver", "customer"): ["loyalty_tier"],
    ("silver", "customer_change"): ["loyalty_tier"],
    ("gold", "dim_customer"): ["loyalty_tier"],
}
print("DRY RUN - nothing will be changed" if DRY_RUN else "LIVE RUN - the selected steps will change data")

In [ ]:
import pandas as pd
CAT = azure_lake.lake_name()
cat = azure_lake.quote_ident(CAT)
lake = azure_lake.connect()   # attaches the Azure DuckLake and makes it the default catalog


def q(sql):
    return lake.execute(sql).fetchall()


def fq(schema, table):
    return f"{cat}.{azure_lake.quote_ident(schema)}.{azure_lake.quote_ident(table)}"


def table_cols(schema, table):
    return [r[0] for r in q(f"SELECT column_name FROM information_schema.columns WHERE table_catalog = '{CAT}' AND table_schema = '{schema}' "
                            f"AND table_name = '{table}' ORDER BY ordinal_position")]


def has_table(schema, table):
    return q(f"SELECT count(*) FROM information_schema.tables WHERE table_catalog = '{CAT}' AND table_schema = '{schema}' AND table_name = '{table}'")[0][0] > 0

## What exists now

In [ ]:
def inventory():
    schemas = [r[0] for r in q(f"SELECT schema_name FROM information_schema.schemata WHERE catalog_name = '{CAT}' AND schema_name IN ('bronze', 'silver', 'gold', 'ops') ORDER BY 1")]
    rows = []
    for s in schemas:
        for t, kind in q(f"SELECT table_name, table_type FROM information_schema.tables WHERE table_catalog = '{CAT}' AND table_schema = '{s}' ORDER BY 2 DESC, 1"):
            rows.append((s, t, kind.lower(), q(f"SELECT count(*) FROM {fq(s, t)}")[0][0], len(table_cols(s, t))))
    return pd.DataFrame(rows, columns=["schema", "name", "type", "rows", "columns"])


inv = inventory()
print(f"{(inv['type'] == 'base table').sum()} tables and {(inv['type'] == 'view').sum()} views in the medallion schemas, {int(inv.loc[inv['type'] == 'base table', 'rows'].sum()):,} table rows" if len(inv) else "no medallion tables found")
inv

In [ ]:
print("Bronze Parquet files in Azure Storage:", q(f"SELECT count(*) FROM glob('{azure_lake.bronze_root()}/*/*/*.parquet')")[0][0])

## Step 1: undo the schema evolution

Drops the evolved columns from the tables that have them. Tables or columns that do not exist (any more) are skipped, so the step can be run again.

In [ ]:
def reset_schema_evolution():
    done = 0
    for (schema, table), columns in EVOLVED_COLUMNS.items():
        if not has_table(schema, table):
            print(f"skip   {schema}.{table}: table does not exist")
            continue
        have = table_cols(schema, table)
        for column in columns:
            if column not in have:
                print(f"skip   {schema}.{table}.{column}: column does not exist")
                continue
            sql = f"ALTER TABLE {fq(schema, table)} DROP COLUMN {azure_lake.quote_ident(column)}"
            if DRY_RUN:
                print("would  ", sql)
            else:
                q(sql)
                print("dropped", f"{schema}.{table}.{column}")
                done += 1
    return done


if RESET_SCHEMA_EVOLUTION:
    reset_schema_evolution()
else:
    print("step 1 switched off")

## Step 2: drop the layers

`DROP SCHEMA ... CASCADE` for `gold`, `silver`, `bronze` and `ops` (in this order). The Parquet files of the dropped tables stay in Azure Storage until step 4 removes them.

In [ ]:
def reset_layers():
    for schema in ("gold", "silver", "bronze", "ops"):
        exists = q(f"SELECT count(*) FROM information_schema.schemata WHERE catalog_name = '{CAT}' AND schema_name = '{schema}'")[0][0] > 0
        sql = f"DROP SCHEMA IF EXISTS {cat}.{schema} CASCADE"
        if not exists:
            print(f"skip   schema {schema}: does not exist")
        elif DRY_RUN:
            print("would  ", sql)
        else:
            q(sql)
            print("dropped schema", schema)


if RESET_LAYERS:
    reset_layers()
else:
    print("step 2 switched off")

## Step 3: delete the Bronze files

Removes everything below `the folder `bronze` of the storage account` (the test data of `01-medallion-generator.ipynb`). Run the generator again to recreate it.

In [ ]:
def delete_bronze_files():
    files, mb = azure_lake.delete_bronze_files(dry_run=DRY_RUN)
    where = azure_lake.bronze_root()
    if DRY_RUN:
        print(f"would  delete {files} files ({mb:,.0f} MB) below {where}")
        return
    print(f"deleted {files} files ({mb:,.0f} MB) below {where}")


if DELETE_BRONZE_FILES:
    delete_bronze_files()
else:
    print("step 3 switched off")

## Step 4: expire snapshots and delete unreferenced files

DuckLake keeps every snapshot (that is what makes time travel possible) and with it the old data files of replaced or dropped tables. `ducklake_expire_snapshots` forgets all snapshots but the newest one,
`ducklake_cleanup_old_files` then deletes the data files that no snapshot references. **This applies to the whole catalog**, not just to the medallion schemas. With `DRY_RUN` both calls only list what they would do.

In [ ]:
def expire_snapshots():
    snapshots = q(f"SELECT count(*) FROM ducklake_snapshots('{CAT}')")[0][0]
    flag = "true" if DRY_RUN else "false"
    expired = q(f"CALL ducklake_expire_snapshots('{CAT}', older_than => now(), dry_run => {flag})")
    print(("would expire" if DRY_RUN else "expired"), len(expired), "of", snapshots, "snapshots")
    old = q(f"CALL ducklake_cleanup_old_files('{CAT}', cleanup_all => true, dry_run => {flag})")
    print(("would delete" if DRY_RUN else "deleted"), len(old), "unreferenced data files")


if EXPIRE_SNAPSHOTS:
    expire_snapshots()
else:
    print("step 4 switched off")

## What is left

In [ ]:
inventory()